<a href="https://colab.research.google.com/github/tuxlimr/Celery_Preprocessing/blob/master/Gmail_automation_Sqlite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install langchain_groq streamlit langchain-community==0.2.12 --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.5/106.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.0/397.0 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 292.2/292.2 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.9/82.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 k

In [ ]:
from google.colab import userdata
groq_api_key = userdata.get('GROQ_KEY')

In [ ]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    temperature=0,
    groq_api_key=groq_api_key,
    model_name="llama-3.1-70b-versatile"
)

### Prompts

##### Extraction role`, `experience`, `skills`, `company`,and `description`

In [ ]:
prompts = {
    "prompt_1":'''
        ### SCRAPED TEXT FROM WEBSITE:
        {page_data}
        ### INSTRUCTION:
        The scraped text is from dataframe column which is extraction from the linkedin job alerts received emails.
        Your job is to extract the job postings and return them in JSON format containing the
        following keys: `role`, `experience`, `skills`, `company`,and `description`.
        Only return the valid JSON.
        ### VALID JSON (NO PREAMBLE):
        ''',
    "prompt_2":
        """
        ### SCRAPED TEXT FROM WEBSITE:
        {page_data}
        ### INSTRUCTION:
        The scraped text is from the career's page of a website.
        Your job is to extract the job postings and return them in JSON format containing the
        following keys: `role`, `experience`, `skills` and `description`.
        Only return the valid JSON.
        ### VALID JSON (NO PREAMBLE):
        # """,
    "linkedin_message":"""
        ## Job Posted by
        {job_poster}
        ### JOB DESCRIPTION:
        {job_description}

        ### INSTRUCTION:

        You are Ajay, Technical Lead at LTIMindtree Limited, an AI & Software Consulting company.
        You've helped numerous enterprises achieve seamless process integration, scalability, optimization, and cost reduction.

        Technical Skills: [Machine learning, Deep learning, Aws/Azure,  Cloud Experience, Data Engineering, NLP, Computer Vision, Python Experience and Generative Ai ]

        Your job is to write a cold message to the Job poster regarding the job mentioned above describing my capability based on Technical skills required for the job.
        The message should be concise and to the point.
        The message should be written in a professional tone.
        The message should be tailored to the job description and add relevent skill as per the job description.


        Do not provide a preamble.
        Dont add content more than 200 words.
        Must Add CV link "https://drive.google.com/file/d/1Psf027IXUnn16bdnabNSpItrQ4xyIs6l/view?usp=sharing"
        Reply in English only
        Create a Subject Line

        Do not add technical skill which i do not know
        I only know "English" as a professional language and Python as Programming language
        Mention to job poster if they offer visa sponsorship for NON-EU citizens

        """

}

##### Save it in Db

In [ ]:
import sqlite3

def save_to_database(job_poster, job, link, res):
    # Connect to SQLite database (or create it if it doesn't exist)
    conn = sqlite3.connect('job_applications.db')

    # Create a cursor object to interact with the database
    cursor = conn.cursor()

    # Create a table to store job application data
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS job_applications (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        job_poster TEXT,
        job_description TEXT,
        job_url TEXT,
        response TEXT
    )
    ''')

    # Sample data to insert
    job_poster = job_poster
    job_description = job

    if page_data:
      job_url = link
    else:
      job_url = input("Enter the job url: ")
    response = res.content

    # Insert the data into the table
    cursor.execute('''
    INSERT INTO job_applications (job_poster, job_description, job_url, response)
    VALUES (?, ?, ?, ?)
    ''', (job_poster, job_description, job_url, response))

    # Commit the transaction
    conn.commit()

    # Close the connection
    conn.close()

    print("Data saved successfully.")
    from google.colab import drive
    drive.mount('/content/drive')
    # prompt: save database in google drive

    import shutil

    # Specify the source and destination paths
    source_path = '/content/job_applications.db'
    destination_path = '/content/drive/MyDrive/job_applications.db'

    # Copy the file to Google Drive
    shutil.copy(source_path, destination_path)

    print("Database saved to Google Drive successfully.")
    return True


In [ ]:
from langchain_community.document_loaders import WebBaseLoader
link = input("Enter the link: ")
loader = WebBaseLoader(link)
page_data = loader.load().pop().page_content
print(page_data)

Enter the link: https://www.make-it-in-germany.com/en/working-in-germany/job-listings/job/job-12727-RE1374336-S
      PLC specialist at engineering people GmbH Nürnberg                                     Go to main navigation   Go to content area         To the homepage        For skilled workers     For employers            Switch language         Language selection        DE     EN     ES     FR       Short information on:     Bahasa Indonesia     Български     Bosanski     اللغة: العربية     Español (México)     Italiano     한국어     Polski     Português     Português (Brasil)     Românesc     Русский     Shqip     Српски     Türkçe     Tiếng Việt     فارسی         Hide search      DE     EN     ES     FR           Show/Hide search        Menu             To the website of the federal government             Search for topics       Search       Hide search         To the homepage    logo                                                              Working in Germany: the official web

In [ ]:
if not page_data:
  job = input("Enter the job content: ")
  job_poster = input("Enter the job poster: ")
else:
  job = page_data
  job_poster = "make it in germany"
# job_url = input("Enter the job url: ")

from langchain_core.prompts import PromptTemplate
prompt_email = PromptTemplate.from_template(
                prompts["linkedin_message"]
        )

chain_email = prompt_email | llm
res = chain_email.invoke({"job_description": str(job),"job_poster": str(job_poster)})
print(res.content)

save_in_db = save_to_database(job_poster, job, link, res)
if save_in_db:
  print("Saved in db")

Subject: Application for PLC Specialist Position at engineering people GmbH Nürnberg

Dear Ms. Pamela Kramer,

I am writing to express my interest in the PLC Specialist position at engineering people GmbH Nürnberg. As a technical professional with experience in automation and control systems, I believe I can contribute to the development of SPS control programs for industrial plants and machines.

With my background in automation and control systems, I am confident in my ability to analyze and implement customer requirements, program and commission complex production systems, and diagnose and troubleshoot existing SPS programs. I am also proficient in Python programming language.

I am particularly drawn to this role because of the opportunity to work on complex automation projects and contribute to the development of high-quality control systems. I am excited about the prospect of joining a team of experienced professionals and contributing to the company's success.

I would like to i

### Send Gmail

In [ ]:
# Install required libraries
# !pip install google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client

import os
import json
import base64
from email.mime.text import MIMEText
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# Define the scopes and base directory
SCOPES = ['https://www.googleapis.com/auth/gmail.send']
BASE_DIRECTORY = '/content/'  # Use Colab's content directory

def authenticate_gmail():
    creds = None
    if os.path.exists(BASE_DIRECTORY + "token.json"):
        creds = Credentials.from_authorized_user_file(BASE_DIRECTORY + "token.json", SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                BASE_DIRECTORY + "credentials.json", SCOPES
            )
            # Use a local server with a defined port for the callback
            creds = flow.run_local_server(port=8080)

        with open(BASE_DIRECTORY + "token.json", "w") as token:
            token.write(creds.to_json())
    return creds

def send_email(creds):
    service = build('gmail', 'v1', credentials=creds)

    message = MIMEText('This is the body of the email')
    message['to'] = 'itsecty.ajay@gmail.com'
    message['subject'] = 'Test Email'
    create_message = {'raw': base64.urlsafe_b64encode(message.as_bytes()).decode()}

    try:
        send_message = (service.users().messages().send(userId="me", body=create_message).execute())
        print(F'Message Id: {send_message["id"]}')
    except HttpError as error:
        print(F'An error occurred: {error}')
        send_message = None

# Upload credentials.json to Colab's content directory
from google.colab import files
uploaded = files.upload()

# Authenticate and send email
creds = authenticate_gmail()
send_email(creds)

Saving credentials.json to credentials (5).json


OSError: [Errno 98] Address already in use